# M3 CLV 조건부 구매취향 이웃 선택 - Dunnhumby

이 노트북은 **구매취향이 비슷한 사용자 후보 안에서 historical CLV 수준과 N/V 구성이 비슷한 사용자를 최종 이웃으로 선택하면 신규상품 추천이 개선되는가**를 확인합니다.

- 학습기간의 이진 구매 TF-IDF cosine으로 사용자별 취향 후보 100명을 먼저 고정합니다.
- historical CLV proxy는 장바구니 수 `N_hat`과 평균 장바구니 금액 `V_hat`의 곱입니다.
- CLV 총수준 `percentile(N_hat × V_hat)`과 N/V 구성좌표를 함께 사용해 후보 안의 최종 이웃 20명을 선택합니다.
- 이력이 짧은 사용자는 `고유상품 수/(고유상품 수+5)`로 CLV 조건을 자동 축소합니다.
- CLV는 이웃 메시지 총량을 늘리지 않고, 같은 취향 후보 중 **누구의 상품정보를 받을지**만 바꿉니다.
- 실제 CLV는 취향-only, degree 층 내 CLV 튜플 shuffle, degree 유사성 대조군과 비교합니다.
- M1의 이진 사용자–상품 전파와 상품 표현은 유지하고 사용자 2층에만 `gamma=0.075`로 이웃 메시지를 결합합니다.
- uniform negative sampling, plain BPR, 하나의 optimizer, 고정 100 epoch를 사용하며 외부 재정렬·표본가중·추가 손실·holdout은 없습니다.

먼저 DAY 1--683 내부의 5개 비중첩 7일 창에서 Candidate Recall@100 사전 관계 진단을 실행합니다. 이 진단이 통과할 때만 DAY 1--683 학습, DAY 684--690 단일 seed 탐색평가를 수행합니다. 단일 seed 결과이므로 유의성이나 일반화를 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, shutil, subprocess, sys

REVIEWED_SHA = '339ff562ea17932b29b1b4061dc81ea46ff794f5'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
%cd /content/clv-m2-lightgcn-runner

# 수정 전 모듈이 Colab 메모리에 남아 재사용되지 않도록 모두 제거합니다.
for module_name in list(sys.modules):
    if module_name.startswith(('lightgcn_clv', 'clv_m3', 'clv_run_state')):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()
print('Pinned execution source:', actual_sha)

In [ ]:
import json
from lightgcn_clv_m3_clv_taste_neighbor_diagnostic import (
    configure_clv_taste_neighbor_diagnostic,
    run_clv_taste_neighbor_mechanism_diagnostic,
)

diagnostic_cfg = configure_clv_taste_neighbor_diagnostic()
mechanism_df = run_clv_taste_neighbor_mechanism_diagnostic(diagnostic_cfg)
MECHANISM_READING = mechanism_df.attrs['reading']
print('\n사전 관계 진단 판정:')
print(json.dumps(MECHANISM_READING, ensure_ascii=False, indent=2))

In [ ]:
import torch
from lightgcn_clv_m3_clv_taste_neighbor import (
    configure_clv_taste_neighbor_run,
    preflight_summary,
    run_clv_taste_neighbor_screen,
)

result_df = None
if not MECHANISM_READING['precheck_passed']:
    print('사전 관계 진단이 통과되지 않아 4개 M3 arm의 고비용 학습을 실행하지 않습니다.')
else:
    assert torch.cuda.is_available(), 'Colab 런타임에서 GPU를 선택한 뒤 이 셀을 다시 실행하세요.'
    cfg = configure_clv_taste_neighbor_run()
    summary = preflight_summary(cfg)
    assert summary['seed'] == 42
    assert summary['historical_development_split']['holdout_constructed'] is False
    assert summary['m3']['historical_clv_proxy'] == 'N_hat * V_hat'
    assert summary['m3']['preference_candidate_neighbors'] == 100
    assert summary['m3']['final_neighbors'] == 20
    assert summary['m3']['reliability_kappa'] == 5.0
    assert summary['m3']['gamma'] == 0.075
    assert summary['fixed']['negative_sampling'] == 'uniform'
    assert summary['fixed']['sample_weighting'] is False
    assert summary['fixed']['one_training_loop_and_optimizer'] is True
    assert summary['fixed']['min_item_interactions'] == 1
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    result_df = run_clv_taste_neighbor_screen(
        cfg, mechanism_reading=MECHANISM_READING
    )

In [ ]:
from IPython.display import display

print('사전진단 CLV 분위·시점별 Candidate Recall@100:')
display(mechanism_df.attrs['summary'])
print('\n사전진단 결과 파일:', mechanism_df.attrs['result_paths'])

if result_df is not None:
    columns = [
        'model_id', 'role',
        'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
        'recall@50', 'ndcg@50',
        'price_purchase_amount_weighted_hit@10',
        'price_purchase_amount_weighted_hit@20',
        'price_purchase_amount_weighted_hit@50',
        'mean_recommended_price_percentile@10',
        'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
        'eff_catalog@10', 'top10_share@10', 'top100_share@10',
    ]
    available = [column for column in columns if column in result_df.columns]
    display(result_df[available])
    print('\nM3 CLV 귀속 판정:')
    print(json.dumps(result_df.attrs['attribution_reading'], ensure_ascii=False, indent=2))
    print('\n그래프 불변조건 진단:')
    print(json.dumps(result_df.attrs['graph_diagnostics'], ensure_ascii=False, indent=2))
    print('\n결과 파일:', result_df.attrs['result_paths'])